# FTIR ML Solver v3 — Colab Training

## Before running
1. **Runtime → Change runtime type → GPU (T4)**
2. Run cells top to bottom

In [ ]:
# Cell 1 — Download repo and install
import urllib.request, zipfile, os, shutil, sys

url = 'https://github.com/DrSuppe/FTIR_absorbtion_ML/archive/refs/heads/main.zip'
print('Downloading...')
urllib.request.urlretrieve(url, '/content/repo.zip')

print('Extracting...')
with zipfile.ZipFile('/content/repo.zip') as z:
    z.extractall('/content/')

extracted = '/content/FTIR_absorbtion_ML-main'
target    = '/content/ftir'
if os.path.exists(target):
    shutil.rmtree(target)
os.rename(extracted, target)
os.chdir(target)
print('Working directory:', os.getcwd())

# Install WITHOUT overwriting Colab's pre-installed torch/numpy
# (do NOT use requirements-pinned.txt — it has versions that conflict with Colab)
!pip install . --no-deps -q

# Colab kernels don't always refresh sys.path immediately after pip install.
# Adding src/ to sys.path guarantees it imports immediately.
sys.path.insert(0, '/content/ftir/src')

# Verify
from ftir_analysis.constants import DEFAULT_TARGET_SPECIES
print('ftir_analysis installed OK')
print('Target species:', DEFAULT_TARGET_SPECIES)

In [ ]:
# Cell 2 — Build / Rebuild Manifest
from ftir_analysis.manifesting import build_manifest
m = build_manifest()
print(f"Manifest: {len(m)} spectra, {m.species.nunique()} unique species")

In [ ]:
# Cell 3 — Quick smoke test (~10 min on T4)
!python3 train.py \
    --n-synthetic 5000 \
    --epochs 10 \
    --batch-size 64 \
    --reference-weight 0.1 \
    --active-label-weight 2.0 \
    --inactive-label-weight 1.0 \
    --log-every 2

In [ ]:
# Cell 4 — Full training run (T4, ~1-2 hrs)
# Uncomment when ready.
# !python3 train.py \
#     --n-synthetic 50000 \
#     --epochs 100 \
#     --batch-size 128 \
#     --lr 3e-4 \
#     --warmup-epochs 5 \
#     --log-every 5

In [ ]:
# Cell 5 — [Optional] Mount Google Drive to persist checkpoints
# from google.colab import drive
# drive.mount('/content/drive')
# Add --checkpoint-dir /content/drive/MyDrive/ftir_checkpoints to train.py

In [ ]:
# Cell 6 — View the latest per-species MAE plot
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.image as mpimg

plots = sorted(Path('outputs/reports').glob('mae_per_species_*.png'))
if plots:
    img = mpimg.imread(plots[-1])
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'MAE — {plots[-1].name}')
    plt.show()
else:
    print('No MAE plots yet — run training first.')

In [ ]:
# Cell 7 — Download best checkpoint
from google.colab import files
files.download('outputs/checkpoints/best_model.pt')